# Train Sunbae on Colab (GPU)

Before running:
1. Runtime -> Change runtime type -> **GPU** (T4 is fine).
2. Upload `dataset/train.bin`, `dataset/val.bin` and `dataset/meta.json`
   to Google Drive: `MyDrive/sunbae/dataset/`.
3. The code must be pushed to GitHub (branch `asadbek`).

The dataset is copied to `/content/dataset/` (fast local disk) before training.
Checkpoints go to `MyDrive/sunbae/checkpoints/`, so they survive a disconnect.
After a disconnect, just run all cells again: training resumes from `last.pt`.

In [ ]:
# Check that a GPU is attached
!nvidia-smi

In [ ]:
# Mount Google Drive (data in, checkpoints out)
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

REPO_URL = "https://github.com/agus787-dev/sunbae"
BRANCH = "asadbek"
REPO_DIR = "/content/sunbae"
DATA_DIR = "/content/drive/MyDrive/sunbae/dataset"   # dataset on Drive
LOCAL_DATA_DIR = "/content/dataset"                  # fast local copy for training
OUT_DIR = "/content/drive/MyDrive/sunbae/checkpoints"  # checkpoints stay on Drive

# Stop early if the dataset is missing on Drive
for name in ["train.bin", "val.bin", "meta.json"]:
    path = os.path.join(DATA_DIR, name)
    assert os.path.exists(path), f"missing {path} - upload dataset/ to Drive first"
    print(f"{name:<10} {os.path.getsize(path) / 1e6:>8.1f} MB")

In [ ]:
# Copy the dataset from Drive to the local disk. Training reads many small
# random windows, which is slow on the Drive mount. Skips files already copied.
import shutil

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
for name in ["train.bin", "val.bin", "meta.json"]:
    src, dst = os.path.join(DATA_DIR, name), os.path.join(LOCAL_DATA_DIR, name)
    if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
        print(f"{name:<10} already copied")
    else:
        shutil.copy(src, dst)
        print(f"{name:<10} copied")

In [ ]:
# Clone the repo (or update it if this runtime already has it)
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -1

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# All 4 checks must pass before spending GPU time
!python tests/test_model.py

In [ ]:
# Train. If last.pt exists on Drive, resume from it automatically.
last_pt = os.path.join(OUT_DIR, "last.pt")
resume = f"--resume {last_pt}" if os.path.exists(last_pt) else ""
print("Resuming from", last_pt if resume else "nothing - starting a new run")

!python scripts/train.py --config configs/run_01.yaml \
    --data_dir {LOCAL_DATA_DIR} --out_dir {OUT_DIR} {resume}